## Simulation of a Floquet discrete time crystal

Adapted from Mi, X., Ippoliti, M., Quintana, C. *et al.* “Time-crystalline eigenstate order on a quantum processor.” *Nature* **601**, 531–536 (2022).  
[https://doi.org/10.1038/s41586-021-04257-w](https://doi.org/10.1038/s41586-021-04257-w)

A crystal spontaneously breaks a symmetry of the system. As in the spontaneous symmetry breaking of the quantum Ising model in HW02, the Hamiltonian describing the atoms in a solid is invariant under arbitrary spatial translations, while the crystalline state is invariant only under the discrete translations of the lattice: continuous translation symmetry is *spontaneously broken*.

In 2012, Wilczek asked whether an analogous phenomenon could occur in **time**: could a many-body system spontaneously develop periodic motion and thereby break time-translation symmetry? For systems in thermal equilibrium, the answer turned out to be no (Bruno 2013; Watanabe & Oshikawa 2015). A loophole appears, however, when the system is driven periodically. A periodically driven, or Floquet, system is invariant only under discrete time translations $t \to t+T$. This discrete symmetry can itself be spontaneously broken if the system responds with a longer period, for example $2T$. Such a phase is called a **discrete time crystal** (DTC) (Khemani *et al.* 2016; Else, Bauer & Nayak 2016). DTC behavior has since been observed in trapped ions, NV centers in diamond, and superconducting quantum processors.

Two ingredients distinguish a genuine time crystal from simple fine-tuned oscillations. First, a generic periodically driven many-body system absorbs energy from the drive and eventually heats toward an infinite-temperature state, where no long-range order survives. A stable time crystal must therefore avoid this heating. Second, the period-$2T$ response must be **rigid**: it should persist when the drive is slightly detuned from the ideal value, rather than occurring only at a finely tuned point. **Many-body localization** (MBL) in a disordered interacting chain can provide both ingredients: localization prevents indefinite heating, while interactions stabilize a collective, rigid period-doubled response.

We study the model implemented on Google's Sycamore processor in Mi *et al.* (2022): a *kicked Ising chain* of $n$ qubits with open boundary conditions. One period of the drive is described by the Floquet unitary

$$
U_F
=
\underbrace{
e^{-i\sum_{j=0}^{n-1} h_j Z_j}
e^{-i\sum_{j=0}^{n-2} \phi_j Z_j Z_{j+1}}
}_{\text{interactions and longitudinal fields}}
\;
\underbrace{
e^{-i\frac{\pi}{2}(1-\epsilon)\sum_{j=0}^{n-1} X_j}
}_{\text{imperfect }\pi\text{ pulse}} .
$$

Thus, during each period we first apply a global rotation about the $x$ axis — the “kick” — and then evolve under nearest-neighbor Ising interactions and longitudinal $z$ fields. After $k$ periods,

$$
|\psi(k)\rangle = U_F^k |\psi_0\rangle,
\qquad
|\psi_0\rangle = |00\cdots0\rangle ,
$$

where the initial state has all spins polarized along $+z$. We observe the system only **stroboscopically**, at integer values of $k$. Our primary observable is the local magnetization

$$
\langle Z_j(k)\rangle
=
\langle\psi(k)|Z_j|\psi(k)\rangle .
$$

The basic intuition is easiest to see at $\epsilon=0$. In this limit the kick is a perfect $\pi$ pulse,

$$
e^{-i\frac{\pi}{2}X} = -iX,
$$

which flips every spin. The remaining part of $U_F$ is diagonal in the $Z$ basis and therefore only adds phases to a $Z$-basis product state. Starting from $|00\cdots0\rangle$, the spins consequently flip back and forth every period,

$$
\langle Z_j(k)\rangle = (-1)^k .
$$

This already produces period doubling, but only for a trivial and fine-tuned reason.

Now detune the pulse so that $\epsilon\neq0$. Each kick then deviates from a perfect $\pi$ rotation by an angle $\pi\epsilon$. For noninteracting spins, $\phi_j=0$, this error accumulates from period to period, producing beating rather than a rigid subharmonic response. You will derive this behavior in Problem 2. In that case the spins simply follow the imperfect drive, and there is no genuine time-crystalline order.

The DTC regime behaves differently. In the presence of **interactions and disorder** in $\phi_j$ and $h_j$, the Floquet system can become many-body localized. The interacting localized system stabilizes the period-doubled response against small pulse errors, so that

$$
\langle Z_j(k)\rangle
\approx
(-1)^k \times \text{constant}
$$

for a finite range of $\epsilon$, rather than only at the perfectly tuned point $\epsilon=0$. By contrast, in a *clean* interacting chain with uniform $\phi_j$ and $h_j$, the system can thermalize under the periodic drive: the magnetization decays toward zero while the entanglement entropy grows rapidly.

Our diagnostics will be the stroboscopic magnetization $\langle Z_j(k)\rangle$, its “unfolded” version

$$
(-1)^k \langle Z_j(k)\rangle,
$$

which is approximately constant in an ideal period-doubled phase, and its discrete Fourier transform over $K$ periods,

$$
\tilde Z_j(\omega)
=
\sum_{k=0}^{K-1}
e^{-i\omega k}
\langle Z_j(k)\rangle,
\qquad
\omega = \frac{2\pi m}{K},
\quad
m=0,\ldots,K-1 .
$$

Period doubling appears as a peak at

$$
\omega=\pi,
$$

corresponding to **half the drive frequency**, or equivalently a **subharmonic response**. Alongside these observables, we will track the half-chain entanglement entropy $S(k)$ of $|\psi(k)\rangle$.

### Why matrix product states?

The Floquet unitary $U_F$ is already a quantum circuit: it consists of a layer of single-site $x$ rotations, a layer of nearest-neighbor $ZZ$ gates, and a layer of single-site $z$ rotations. The gate machinery developed in Notebook 05 (`psi.gate_` with `contract='swap+split'`) therefore applies directly.

Importantly, there is **no Trotter error** here. Unlike the Hamiltonian time evolution considered in HW03, the Floquet dynamics is itself defined as a sequence of gates. The only numerical approximation comes from truncating the MPS bond dimension to some maximum value $\chi$.

Whether this truncation remains accurate is determined by the growth of entanglement, which is itself central to the physics. In the MBL regime, entanglement grows only slowly — typically logarithmically in time — allowing an MPS with moderate bond dimension to remain accurate for long times. In a thermalizing clean system, by contrast, entanglement grows much more rapidly, eventually making a fixed-$\chi$ MPS description inaccurate.

This makes TEBD/MPS particularly well suited to studying the time-crystal regime: it allows us to simulate chains of $n\sim40$–$100$ qubits for hundreds or even thousands of Floquet periods, far beyond the system sizes accessible to exact diagonalization.

### Parameters and conventions

- **Disordered chain:**

  $$
  \phi_j \sim \mathrm{Uniform}\left[\frac{\pi}{8},\frac{3\pi}{8}\right],
  \qquad
  h_j \sim \mathrm{Uniform}\left[-\frac{\pi}{2},\frac{\pi}{2}\right].
  $$

  These ranges follow Mi *et al.* after translating to our convention. Fix the random-number-generator seed so that your simulations are reproducible, and use the **same disorder realization for all values of $\epsilon$** unless instructed otherwise.

- **Clean chain:**

  $$
  \phi_j = 0.8,
  \qquad
  h_j = 0.4
  $$

  for all $j$. A nonzero longitudinal field $h_j$ is important here. At $h_j=0$, the kicked chain reduces to the kicked transverse-field Ising model, which maps to free fermions, as in HW02, and does not exhibit generic interacting thermalization.

- $X$ and $Z$ denote Pauli matrices, not spin-$\frac{1}{2}$ operators.

- Use the quimb site-ordering convention of NB04/NB05, with site $0$ as the leftmost site.

- Use open boundary conditions throughout.

- **Compute budget:** For $n=40$, $\chi=64$, and measurements of all $\langle Z_j(k)\rangle$ after every period, a 200-period simulation should take roughly a minute on a typical laptop. Measuring observables less frequently — for example, every 10 periods — makes substantially longer runs inexpensive. Increasing the bond dimension from $\chi=64$ to $\chi=128$ increases the computational cost by roughly a factor of four.

## Problems (80 pts + 10 pts bonus)

Feel free to use your own code from notebook 05, quimb's `MatrixProductState.gate_`, `qtn.Circuit`, or any other approach. Exact state-vector calculations are welcome as checks for small $n$.

### Rigid period doubling in a kicked Ising chain

1. (10 pts) **Floquet circuit.** Write a function `floquet_period_(psi, eps, phis, hs, max_bond, info)` that applies one period $U_F$ *in place* to a quimb MPS: the kick $e^{-i\frac{\pi}{2}(1-\epsilon)X_j}$ on every site (`contract=True`), then the gates $e^{-i\phi_j Z_jZ_{j+1}}$ on all bonds (`contract='swap+split'`, with `max_bond` and `cutoff=1e-10`), then $e^{-ih_jZ_j}$ on every site. Check your function: for $n = 10$, $\epsilon = 0.05$ and one disordered realization, compare $\langle Z_j(k)\rangle$ for $k \le 20$ with a dense state-vector calculation (hint: $U_F$ is a diagonal matrix of phases times a product of single-qubit rotations, so you never need the $2^n\times 2^n$ matrix) and report the maximum deviation.

2. (15 pts) **Free spins.** Set $\phi_j = h_j = 0$.
   (a) Show analytically that $\langle Z_j(k)\rangle = (-1)^k\cos(\pi\epsilon k)$. (Hint: $e^{-i\theta X}$ rotates the spin about the $x$ axis by the angle $2\theta$; alternatively work in the Heisenberg picture and compute $U_F^\dagger Z_j U_F$.)
   (b) Verify this numerically for $\epsilon = 0.05$ ($n = 12$ is enough). Over how many periods does the signal survive, and how does this scale with $\epsilon$?
   (c) Plot $|\tilde Z_{n/2}(\omega)|$ for $K = 200$ and $\epsilon = 0$, $0.03$, $0.06$, $0.1$. Where are the peaks, and how do they move with $\epsilon$? Explain why this is *not* a time crystal.

3. (30 pts) **The disordered chain: a time crystal.** Take $n \ge 40$, $\chi = 64$, $K = 200$ periods and one disordered realization of $\phi_j$, $h_j$. For $\epsilon = 0$, $0.03$, $0.06$, $0.1$:
   (a) Plot a spacetime heatmap of $(-1)^k\langle Z_j(k)\rangle$ (site $j$ on the $x$-axis, period $k$ on the $y$-axis). Describe what you see, including any difference between the bulk and the edge sites.
   (b) Plot $(-1)^k\langle Z_{n/2}(k)\rangle$ versus $k$ together with the free-spin envelope $\cos(\pi\epsilon k)$ of problem 2. Does the magnetization follow the detuning of the drive?
   (c) Plot $|\tilde Z_{n/2}(\omega)|$ for each $\epsilon$ and compare with problem 2(c). Does the peak stay at $\omega = \pi$ when $\epsilon \ne 0$? This is the *rigidity* of the subharmonic response.
   (Optional: increase $\epsilon$ further and estimate where the time crystal melts, i.e. where the $\omega = \pi$ peak disappears.)

4. (25 pts) **Entanglement, heating, and the limits of TEBD.** Compute the half-chain entanglement entropy $S(k)$ at the central bond after every period.
   (a) *Clean chain* ($\phi_j = 0.8$, $h_j = 0.4$), $\epsilon = 0.05$, $K = 100$: run with $\chi = 32$, $64$ and $128$ and plot $S(k)$ and $(-1)^k\langle Z_{n/2}(k)\rangle$ for the three bond dimensions on the same axes. Describe the growth of $S(k)$ before truncation sets in. At which $k$ does each $\chi$ stop being converged, and how is the value at which $S$ saturates related to $\chi$ (hint: at most $\log_2\chi$ bits can fit through a bond)? Does the magnetization decay — does the clean chain heat?
   (b) *Disordered chain*, $\epsilon = 0.1$, $\chi = 64$, $K = 1000$ periods (measure every 10 periods; add $\chi = 128$ if you have the patience): plot $S(k)$ against $k$ on a log-linear scale (logarithmic $k$ axis). How does the entanglement grow, and how does it compare with (a)? Why can TEBD reach $1000$ periods here but not in (a)? Relate your answer to many-body localization and to the absence of heating that a time crystal requires.

5. (10 pts, bonus) **Disorder average.** In the experiment, the signal is averaged over many disorder realizations. For $\epsilon = 0.1$, $n = 40$, $K = 300$ periods, run about 5 realizations of $\phi_j, h_j$ and plot the average of $(-1)^k\langle Z_j(k)\rangle$ over the bulk sites and over realizations, together with the curves of the individual realizations. How large are the realization-to-realization fluctuations compared with the averaged signal, and does the average still look locked to $(-1)^k$?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm
import quimb as qu
import quimb.tensor as qtn

X = qu.pauli('X').real
Z = qu.pauli('Z').real
ZZ = np.kron(Z, Z)


def floquet_period_(psi, eps, phis, hs, max_bond, info):
    '''Apply one Floquet period U_F in place to the MPS `psi`.

    eps  : detuning of the pi pulse
    phis : array of n-1 Ising couplings phi_j
    hs   : array of n longitudinal fields h_j
    '''
    pass


def measure_Z(psi):
    '''Return the array <Z_j> for all sites j.'''
    pass


#### Your code here. Insert more cells if needed.
